# Qwen3.5-4B 答案优先格式软标签蒸馏训练

基于本地已有的训练脚本和数据集，在 Colab 免费 T4 GPU 上运行

**数据格式**: 答案优先 `{"sentiment": X}`
**训练方法**: 软标签蒸馏 (KL + SFT 混合损失)

| 配置 |  |
|------|-----|
| 数据量 | 7172 条 |
| Epochs | 3 |
| 预计时间 | ~2小时 |
| GPU | T4 (16GB) |

**对比目标**: 已有 Qwen3-4B 固定温度结果 80.38%

In [ ]:
#@title 1. 安装依赖
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy; _numpy = f'numpy=={numpy.__version__}'
    except: _numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    # 纯文本训练，移除 torchvision
    !uv pip install -qqq --upgrade {_vllm} {_numpy} bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}

!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

print("✓ 依赖安装完成")

In [ ]:
#@title 2. 上传训练数据
from google.colab import files
import json

print("请上传 train_answer_first.json 文件:
uploaded = files.upload()

# 保存文件
for fn in uploaded.keys():
    print(f'上传文件: {fn} ({len(uploaded[fn])} bytes)')

# 验证数据格式
with open('train_answer_first.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"\n数据量: {len(data)} 条")
print(f"格式检查: {data[0]['conversations'][2]['content'][:30]}...
print(f"软标签: {data[0].get('soft_labels', '无')}")
print("\n✓ 数据验证完成")

In [ ]:
#@title 3. 训练配置
import torch
import torch.nn.functional as F
import json
import numpy as np
from pathlib import Path

# 配置参数
MAX_SEQ_LENGTH = 512  #@param {type:"integer"}
LORA_RANK = 16  #@param {type:"integer"}
EPOCHS = 3  #@param {type:"integer"}
TEMPERATURE = 2.0  #@param {type:"number"}
ALPHA = 0.5  #@param {type:"number"}
USE_DYNAMIC_TEMP = False  #@param {type:"boolean"}

RANDOM_STATE = 3407

print(f"配置确认:")
print(f"  MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"  LORA_RANK: {LORA_RANK}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  TEMPERATURE: {TEMPERATURE}")
print(f"  ALPHA: {ALPHA}")
print(f"  USE_DYNAMIC_TEMP: {USE_DYNAMIC_TEMP}")

In [ ]:
#@title 4. 加载模型
from unsloth import FastLanguageModel

print("加载模型: unsloth/Qwen3.5-4B")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",  # 使用 unsloth 版本避免 VL processor
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,  # Colab T4 用 16-bit
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)

# LoRA 配置
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    random_state=RANDOM_STATE,
)

# 显示 GPU 内存状态
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"\nGPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
print("\n✓ 模型加载完成")

In [ ]:
#@title 5. 数据预处理
# 动态温度策略
def adaptive_temperature(confidence: float) -> float:
    if confidence > 0.9:
        return 1.5
    elif confidence > 0.6:
        return 2.0
    else:
        return min(2.5 + (0.6 - confidence) * 2, 3.0)

# Sentiment token IDs
sentiment_ids = [
    tokenizer.encode('0', add_special_tokens=False)[0],
    tokenizer.encode('1', add_special_tokens=False)[0],
    tokenizer.encode('2', add_special_tokens=False)[0],
]
print(f"Sentiment token IDs: {sentiment_ids}")

# 加载并预处理数据
with open('train_answer_first.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

records = []
for item in data:
    conv = item['conversations']

    # ChatML 格式
    full_text = tokenizer.apply_chat_template(
        conv, tokenize=False, add_generation_prompt=False
    )
    prefix_text = tokenizer.apply_chat_template(
        conv[:-1], tokenize=False, add_generation_prompt=True
    )

    full_ids = tokenizer.encode(full_text, add_special_tokens=False)[:MAX_SEQ_LENGTH]
    prefix_len = min(len(tokenizer.encode(prefix_text, add_special_tokens=False)), len(full_ids))

    labels = [-100] * prefix_len + full_ids[prefix_len:]

    # 定位 sentiment token
    sentiment_pos = -1
    target_str = '"sentiment": '
    idx = full_text.find(target_str)
    if idx != -1:
        pos = len(tokenizer.encode(full_text[:idx + len(target_str)], add_special_tokens=False))
        if pos < len(full_ids) and full_ids[pos] in sentiment_ids:
            sentiment_pos = pos

    # 置信度
    soft_labels = item.get('soft_labels', [0.33, 0.33, 0.34])
    confidence = max(soft_labels)

    records.append({
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
        'soft_labels': soft_labels,
        'sentiment_pos': sentiment_pos,
        'confidence': confidence,
    })

n_valid = sum(1 for r in records if r['sentiment_pos'] != -1)
print(f"Sentiment 定位成功: {n_valid}/{len(records)}")

# 创建 Dataset
from datasets import Dataset as HFDataset
dataset = HFDataset.from_list(records)

print("\n✓ 数据预处理完成")

In [ ]:
#@title 6. 开始训练
# DataCollator
class DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        soft_labels = [f.pop('soft_labels') for f in features]
        sentiment_pos = [f.pop('sentiment_pos') for f in features]
        confidence = [f.pop('confidence') for f in features]

        max_len = max(len(f['input_ids']) for f in features)
        pad_id = self.tokenizer.pad_token_id or 0

        input_ids = torch.tensor([
            f['input_ids'] + [pad_id] * (max_len - len(f['input_ids']))
            for f in features
        ], dtype=torch.long)

        attention_mask = torch.tensor([
            f['attention_mask'] + [0] * (max_len - len(f['attention_mask']))
            for f in features
        ], dtype=torch.long)

        labels = torch.tensor([
            f['labels'] + [-100] * (max_len - len(f['labels']))
            for f in features
        ], dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'soft_labels': torch.tensor(soft_labels, dtype=torch.float32),
            'sentiment_pos': torch.tensor(sentiment_pos, dtype=torch.long),
            'confidence': torch.tensor(confidence, dtype=torch.float32),
        }

# 自定义 compute_loss
def compute_loss(model, inputs, return_outputs=False, **kwargs):
    soft_labels = inputs.pop("soft_labels", None)
    sentiment_pos = inputs.pop("sentiment_pos", None)
    labels = inputs.pop("labels", None)
    confidences = inputs.pop("confidence", None)

    outputs = model(**inputs)
    logits = outputs.logits

    # SFT Loss
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    if (shift_labels != -100).sum() == 0:
        sft_loss = torch.tensor(0.0, device=logits.device, requires_grad=True)
    else:
        sft_loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

    # KL Loss
    kl_losses = []
    temps_used = []

    if soft_labels is not None and sentiment_pos is not None:
        soft_labels = soft_labels.to(logits.device)

        for b in range(logits.size(0)):
            pos = sentiment_pos[b].item()
            if pos == -1:
                continue

            target_pos = pos - 1
            if target_pos < 0 or target_pos >= logits.size(1):
                continue

            # 温度选择
            if USE_DYNAMIC_TEMP:
                confidence = confidences[b].item() if confidences is not None else 0.5
                temp = adaptive_temperature(confidence)
            else:
                temp = TEMPERATURE
            temps_used.append(temp)

            # KL 计算
            sent_logits = logits[b, target_pos, sentiment_ids].float()
            sent_logits = sent_logits / temp
            student_log_probs = F.log_softmax(sent_logits, dim=-1)
            teacher_probs = soft_labels[b]

            kl = F.kl_div(student_log_probs, teacher_probs, reduction='sum')
            kl = kl * (temp ** 2)
            kl_losses.append(kl)

    if kl_losses:
        kl_loss = torch.stack(kl_losses).mean()
        total_loss = ALPHA * kl_loss + (1 - ALPHA) * sft_loss
    else:
        kl_loss = torch.tensor(0.0, device=logits.device)
        total_loss = sft_loss

    # Debug 输出
    if not hasattr(compute_loss, '_step'):
        compute_loss._step = 0
    if compute_loss._step < 5:
        avg_temp = np.mean(temps_used) if temps_used else TEMPERATURE
        print(f"[{compute_loss._step}] sft={sft_loss.item():.4f} kl={kl_loss.item():.4f} total={total_loss.item():.4f} avg_temp={avg_temp:.2f}")
        compute_loss._step += 1

    return (total_loss, outputs) if return_outputs else total_loss

# 训练配置
n_examples = len(data)
total_steps = (n_examples // 16) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))

from transformers import TrainingArguments, Trainer

output_dir = "qwen35-4b-answer-first"

training_args = TrainingArguments(
    output_dir=output_dir + "_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=warmup_steps,
    num_train_epochs=EPOCHS,
    learning_rate=2e-5,
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=RANDOM_STATE,
    report_to="none",
    bf16=True,
    fp16=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollator(tokenizer),
)

trainer.compute_loss = compute_loss

print(f"\n开始训练:")
print(f"  Epochs: {EPOCHS}")
print(f"  Temperature: {TEMPERATURE if not USE_DYNAMIC_TEMP else 'adaptive (1.5-3.0)'}")
print(f"  Alpha: {ALPHA}")
print(f"  Total steps: {total_steps}")

# 开始训练
trainer_stats = trainer.train()

# 显示训练统计
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\n{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

print("\n✓ 训练完成")

In [ ]:
#@title 7. 保存模型
Path(output_dir).mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# 配置记录
config = {
    "model": "unsloth/Qwen3.5-4B",
    "data": "train_answer_first.json",
    "epochs": EPOCHS,
    "temperature": TEMPERATURE if not USE_DYNAMIC_TEMP else "adaptive",
    "alpha": ALPHA,
    "format": "answer_first",
    "train_time_seconds": trainer_stats.metrics['train_runtime'],
}
with open(Path(output_dir) / "train_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"模型保存到: {output_dir}")
print("\n✓ 模型保存完成")

In [ ]:
#@title 8. 下载模型
from google.colab import files
import shutil

# 打包模型文件
shutil.make_archive(output_dir, 'zip', output_dir)

# 下载
files.download(f'{output_dir}.zip')
print("\n✓ 模型已下载到本地")

## 训练完成

模型已下载到本地，后续步骤：

1. 解压模型文件
2. 使用 `eval_answer_first.py` 进行评估
3. 对比 Qwen3-4B 固定温度结果 (80.38%)

**评估命令**:
```bash
cd 4_evaluation
python3 eval_answer_first.py --model ../3_lora_training/models/qwen35-4b-answer-first
```